Se implementa un sistema interactivo en consola que permite filtrar información según las cedulas o la edad para saber cuantes ventas de vehiculos se han vendido mas, 

El uso de Polars en este caso ofrece un alto rendimiento al trabajar con archivos grandes, optimizando tanto el tiempo de carga como las operaciones de filtrado ya que el archivo principal tiene 1 millon de filas para anlizar como se muestra en la imagen el resultado de una prueba y el rendimiento que tubo polars a la hora de ejcutar esta funcion.
 ![Texto alternativo](../../../../otros-elementos/vente-vehiculos-prueba-elifs-polars.png)

 de esta forma es como se tiene que ver el resultado de la prueba



In [12]:
%%time  

import polars as pl

# Cargar el archivo CSV en un DataFrame de Polars
archivo = pl.read_csv("ventas_vehiculos_simuladas1.csv")

# Punto de entrada principal del script
if __name__ == "__main__":

    # Mostrar el total de registros cargados
    print(f"Total de registros cargados: {len(archivo):,}\n")

    # Menú interactivo para seleccionar el tipo de búsqueda
    print("=== FORMULARIO DE BÚSQUEDA ===")
    print("1. Buscar por cédula")
    print("2. Buscar por edad")
    print("3. Buscar por año de venta")
    print("4. Buscar por número celular")
    print("5. Buscar por valor del vehículo")
    print("6. Buscar por IVA")
    print("7. Buscar por valor total\n")

    # Captura de la opción del usuario
    opcion = input("Elige el tipo de búsqueda (1–7): ").strip()

    # --- BÚSQUEDA POR CÉDULA ---
    if opcion == "1":
        campo = "cedula"  # Columna a filtrar
        objetivo = input("Ingresa la cédula que deseas buscar: ").strip()
        tipo = pl.Utf8    # Tipo string en Polars

    # --- BÚSQUEDA POR EDAD ---
    elif opcion == "2":
        campo = "edad"
        objetivo = input("Ingresa la edad que deseas buscar: ").strip()
        tipo = pl.Int64   # Tipo entero

        # Validación para asegurar que el valor ingresado sea numérico
        try:
            objetivo = int(objetivo)
        except ValueError:
            print("Error: La edad debe ser un número entero.")
            exit()

    # --- BÚSQUEDA POR AÑO DE VENTA ---
    elif opcion == "3":
        campo = "fecha_venta"

        # Solicitar el año
        objetivo = input("Ingresa el año que deseas buscar (por ejemplo, 2023): ").strip()

        # Validar que sea número entero
        try:
            objetivo = int(objetivo)
        except ValueError:
            print("Error: El año debe ser un número entero.")
            exit()

        # Convertir la columna de texto a tipo fecha
        archivo = archivo.with_columns(
            pl.col("fecha_venta")
            .str.strptime(pl.Date, "%Y-%m-%d")  # Parseo de string a fecha
            .alias("fecha_venta")
        )

        # Filtrar por el año extraído de la fecha
        resultado = archivo.filter(
            pl.col("fecha_venta").dt.year() == anio
        )

    # --- BÚSQUEDA POR NÚMERO CELULAR ---
    elif opcion == "4":
        campo = "numero_celular"
        objetivo = input("Ingresa el número celular: ").strip()
        tipo = pl.Int64

        try:
            objetivo = int(objetivo)
        except ValueError:
            print("Error: El número celular debe ser un número entero.")
            exit()

    # --- BÚSQUEDA POR VALOR DEL VEHÍCULO ---
    elif opcion == "5":
        campo = "valor_vehiculo"
        objetivo = input("Ingresa el valor exacto del vehículo: ").strip()
        tipo = pl.Float64  # Tipo decimal

        try:
            objetivo = float(objetivo)
        except ValueError:
            print("Error: El valor debe ser un número decimal o entero.")
            exit()

    # --- BÚSQUEDA POR IVA ---
    elif opcion == "6":
        campo = "iva"
        objetivo = input("Ingresa el valor exacto del IVA: ").strip()
        tipo = pl.Int64

        try:
            objetivo = int(objetivo)
        except ValueError:
            print("Error: El IVA debe ser un número entero.")
            exit()

    # --- BÚSQUEDA POR VALOR TOTAL ---
    elif opcion == "7":
        campo = "valor_total"
        objetivo = input("Ingresa el valor total: ").strip()
        tipo = pl.Float64

        try:
            objetivo = float(objetivo)
        except ValueError:
            print("Error: El valor total debe ser un número decimal o entero.")
            exit()

    # --- OPCIÓN INVÁLIDA ---
    else:
        print("Opción no válida. Saliendo...")
        exit()

    # --- FILTRADO GENERAL ---
    # Se aplica a todas las opciones excepto la búsqueda por año (ya filtrada arriba)
    if opcion in ["1", "2", "4", "5", "6", "7"]:
        resultado = archivo.filter(
            pl.col(campo).cast(tipo) == objetivo
        )

    # --- MOSTRAR RESULTADOS ---
    if resultado.height > 0:
        print(f"Se encontraron {resultado.height} coincidencias para {campo} = {objetivo}\n")
        print(resultado)  # Muestra el DataFrame filtrado
    else:
        print("No se encontraron resultados para tu búsqueda.")

Total de registros cargados: 10,000

=== FORMULARIO DE BÚSQUEDA ===
1. Buscar por cédula
2. Buscar por edad
3. Buscar por año de venta
4. Buscar por número celular
5. Buscar por valor del vehículo
6. Buscar por IVA
7. Buscar por valor total

No se encontraron resultados para tu búsqueda.
CPU times: user 32.2 ms, sys: 6.77 ms, total: 39 ms
Wall time: 2.38 s


## 📥 Uso de dataset completo (1M de registros)

Para trabajar con el dataset completo (más de 1 millón de registros),
puedes descargar el archivo `.csv` desde la siguiente carpeta:

👉 [Carpeta con
datos](https://drive.google.com/drive/folders/1tuugYsOuOAbYCUeusr2hhc7JakumANPW)

Dentro de esta carpeta encontrarás el archivo:

-   **venta-vehiculos-simuladas.csv**

------------------------------------------------------------------------

## ⚙️ Instrucciones

Una vez descargado, debes ubicar el archivo dentro de la carpeta del
proyecto correspondiente:

``` bash
venta-de-vehiculos/
├── analisis.ipynb
├── venta-vehiculos-simuladas.csv
```

Esto es necesario para que el notebook pueda acceder correctamente al
archivo y evitar errores en las rutas de lectura.


# ⚠️ Importante

para poder uar bien el codigo con el otro csv hay que cambiar la ruta del archivo que estamos usando ya que el archivo reducido tiene un nombre diferente al original en este caso etos son los nombres 

- archivo con poco datos: **ventas_vehiculos_simuladas1.csv**
- archivo con millones de dato: **ventas_vehiculos_simuladas.csv**

en este caso en la linea donde se debe cambiar eto es en esta
``` bash
archivo = pl.read_csv("ventas_vehiculos_simuladas.csv")
``` 